In [0]:
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Config
CATALOG = "oil_gas_demo"
BRONZE = f"{CATALOG}.bronze.supply_chain_transactions"
SILVER = f"{CATALOG}.silver.supply_chain_transactions"


class SilverTransformer:
    """Transforms bronze into silver by removing duplicates."""

    def __init__(self, spark, source_table: str, target_table: str):
        self.spark = spark
        self.source_table = source_table
        self.target_table = target_table
        self.df = None

    def load_bronze(self):
        """Read the bronze table."""
        logger.info(f"Loading bronze: {self.source_table}")
        self.df = self.spark.table(self.source_table)
        logger.info(f"Loaded {self.df.count()} rows")

    def remove_duplicates(self):
        """Keep the latest record per transaction_id."""
        logger.info("Removing duplicates by transaction_id")
        window = Window.partitionBy("transaction_id").orderBy(F.col("_ingest_ts").desc())
        self.df = (
            self.df
            .withColumns({"_row_num": F.row_number().over(window)})
            .filter("_row_num = 1")
            .drop("_row_num")
        )

    def write_to_silver(self):
        """Write to the silver Delta table."""
        count = self.df.count()
        logger.info(f"Writing {count:,} rows to silver: {self.target_table}")
        self.df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(self.target_table)
        logger.info(f"Silver table written: {self.target_table} ({count:,} rows)")

    def run(self):
        """Load -> dedupe -> write."""
        self.load_bronze()
        self.remove_duplicates()
        self.write_to_silver()


# Execute
silver = SilverTransformer(
    spark=spark,
    source_table=BRONZE,
    target_table=SILVER,
)
silver.run()